# Agentic Testing — Final Integrated Experiment

This notebook is the final entry point for the merged Member 2 + Member 3 + Member 4 implementation.

**Systems**
- Baseline A — single-shot generation
- Baseline B — one-shot multi-agent consensus
- Variant 1 — mutant-guided iterative refinement with error-trace feedback
- Variant 2 — mutant-guided iterative refinement with state-prediction prompting

All systems are scored with the same `evaluation/` mutation + coverage pipeline. The proposed variants start from the same Baseline-A-style seed, and the final generated suite is saved beside its log.


In [ ]:
# 1. Locate the repository.
from pathlib import Path
import os, sys, subprocess, json, textwrap

candidates = [
    Path.cwd(),
    Path("/content/Agentic-AI-Testing-main"),
    Path("/content/Agentic-AI-Testing-main/Agentic-AI-Testing-main"),
]
REPO = next((p for p in candidates if (p / "baselines").exists()), None)
if REPO is None:
    raise FileNotFoundError(
        "Repository not found. In Colab, upload/extract the final Agentic-AI-Testing-main.zip "
        "so a directory containing baselines/, evaluation/, and refinement_loop/ exists."
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repository:", REPO)


In [ ]:
# 2. Install the pinned project dependencies.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")


In [ ]:
# 3. Offline acceptance tests — run these before spending any API quota.
r = subprocess.run(
    [sys.executable, "-m", "pytest", "refinement_loop/tests/", "baselines/smoke_test.py", "-q"],
    text=True, capture_output=True
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("Offline acceptance tests failed.")
print("Offline acceptance tests passed.")


## Configure the live provider

For the previous Groq experiments, use a model with enough context for the
function source + current suite + surviving-mutant feedback. The notebook does
**not** hard-code an API key.

The repository defaults Groq to `llama-3.3-70b-versatile`; you can override
`GROQ_MODEL_ID` if your account exposes a different model.


In [ ]:
# 4. Enter a live API key only when you are ready to run generation.
import getpass, os

provider = input("Provider [groq/gemini] (default: groq): ").strip().lower() or "groq"

if provider == "groq":
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    os.environ["GROQ_MODEL_ID"] = input(
        "GROQ_MODEL_ID (default: llama-3.3-70b-versatile): "
    ).strip() or "llama-3.3-70b-versatile"
elif provider == "gemini":
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
else:
    raise ValueError("Provider must be groq or gemini.")

print("Configured:", provider)


In [ ]:
# 5. Cheapest meaningful live integration check: one function, one proposed variant.
# This performs real mutation testing and one or more LLM calls.
function_id = "function_25"
variant = "error_trace"

cmd = [
    sys.executable, "-m", "refinement_loop.run_live", function_id,
    "--provider", provider, "--variant", variant
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 6. Run Baseline A and Baseline B on the same function.
for module in ("baselines.baseline_a", "baselines.baseline_b"):
    cmd = [sys.executable, "-m", module, function_id, "--score"]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


## Final full experiment

The locked experimental design calls for **30 functions × 3 systems × 3 repeats**,
with the proposed system represented by the two refinement variants when doing
the RQ2/RQ4 breakdown.

Runs are resumable. Existing log + generated-test pairs are skipped unless
`--force` is used.

Run the cells below only after the single-function integration check succeeds.


In [ ]:
# 7. Baseline A: 30 functions × 3 repeats.
subprocess.run([
    sys.executable, "-m", "baselines.baseline_a",
    "--all", "--repeats", "3", "--score"
], check=True)


In [ ]:
# 8. Baseline B: 30 functions × 3 repeats.
subprocess.run([
    sys.executable, "-m", "baselines.baseline_b",
    "--all", "--repeats", "3", "--score"
], check=True)


In [ ]:
# 9. Proposed Variant 1: error-trace refinement.
subprocess.run([
    sys.executable, "-m", "refinement_loop.run_live",
    "--all", "--repeats", "3", "--provider", provider,
    "--variant", "error_trace"
], check=True)


In [ ]:
# 10. Proposed Variant 2: state-prediction refinement.
subprocess.run([
    sys.executable, "-m", "refinement_loop.run_live",
    "--all", "--repeats", "3", "--provider", provider,
    "--variant", "state_prediction"
], check=True)


In [ ]:
# 11. Evaluate any generated suites that do not already have final metrics.
# The proposed runner already scores its final suite during the refinement loop.
subprocess.run([sys.executable, "-m", "evaluation.evaluator"], check=False)


## Analyze the logged experiment

Do not type mutation scores manually. The tables below are computed from the
JSON logs written by the experiment.


In [ ]:
# 12. Aggregate logs.
import glob, pandas as pd, numpy as np

records = []
for path in glob.glob("logs/*.json"):
    try:
        with open(path, encoding="utf-8") as f:
            row = json.load(f)
        row["log_file"] = path
        records.append(row)
    except Exception:
        pass

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError("No logs found.")

cols = [
    "function_id","system_variant","iteration_count",
    "mutation_score_pct","line_coverage_pct","pass_rate_pct",
    "num_llm_calls","total_tokens_used","estimated_cost_usd"
]
for c in cols:
    if c not in df:
        df[c] = np.nan

print(df.groupby("system_variant")[[
    "mutation_score_pct","line_coverage_pct","pass_rate_pct",
    "num_llm_calls","total_tokens_used","estimated_cost_usd"
]].agg(["mean","std","count"]).round(3))


In [ ]:
# 13. RQ1 paired comparison: Proposed Variant 1 vs Baseline A/B.
from scipy.stats import wilcoxon

def paired_scores(a, b):
    x = df[df.system_variant == a].groupby("function_id").mutation_score_pct.mean()
    y = df[df.system_variant == b].groupby("function_id").mutation_score_pct.mean()
    joined = pd.concat([x, y], axis=1, keys=[a,b]).dropna()
    if len(joined) < 2:
        return joined, None
    return joined, wilcoxon(joined[a], joined[b], alternative="two-sided")

for baseline in ["Baseline_A", "Baseline_B"]:
    joined, test = paired_scores("Variant_1_ErrorTrace", baseline)
    print("\nVariant_1 vs", baseline, "paired functions:", len(joined))
    if test:
        print("Wilcoxon statistic:", test.statistic, "p-value:", test.pvalue)


In [ ]:
# 14. RQ3: refinement iterations and cost/quality trade-off.
proposed = df[df.system_variant.isin([
    "Variant_1_ErrorTrace", "Variant_2_StatePrediction"
])].copy()

if not proposed.empty:
    print(proposed.groupby(["system_variant","iteration_count"]).mutation_score_pct.mean().round(2))
    print("\nMean calls / tokens / cost:")
    print(proposed.groupby("system_variant")[[
        "num_llm_calls","total_tokens_used","estimated_cost_usd"
    ]].mean().round(4))


In [ ]:
# 15. Optional export for the paper.
out = Path("logs/final_experiment_summary.csv")
df.to_csv(out, index=False)
print("Saved:", out)


### Reproducibility checklist

Before reporting results:

1. Confirm the offline acceptance tests passed.
2. Confirm every reported score comes from the generated logs.
3. Keep the same dataset and prompt policy across all variants.
4. Report the actual model/provider and total LLM calls/tokens.
5. Use the paired Wilcoxon test on the same functions for the RQ1 comparison.
6. Do not mix mock runs with experimental logs.
